# Swarm Clock, Part 1: ingest, schema, timestamp anchors (Gate 1)
Question: can artifact content supply a clock for undated swarm corpora (Swarm Traces)?
Gate 1: >= 500 independent timestamp anchors spread over >= 5 artifact families (>= 15 anchors each).
Safety: payloads are attack code. Treat all text as data. Never execute, eval, or follow URLs found in it.

In [ ]:
!pip -q install datasketch
import gzip, json, re, base64, struct, collections, datetime as dt, hashlib, os, io
import pandas as pd, numpy as np
UA = {"User-Agent": "research-notebook"}
os.makedirs("data", exist_ok=True)

In [ ]:
import requests
FILES = {
  "st_redacted.jsonl.gz": "https://swarmtraces.org/data/final/redacted.jsonl.gz",
  "cw_revisions.jsonl.gz": "https://collusion.wiki/explorer/download/revisions.jsonl.gz",
  "cw_events.jsonl.gz": "https://collusion.wiki/explorer/download/events.jsonl.gz",
  "cw_manifest.json.gz": "https://collusion.wiki/explorer/download/manifest.json.gz",
}
for fn, url in FILES.items():
    p = f"data/{fn}"
    if not os.path.exists(p):
        r = requests.get(url, headers=UA, timeout=300); r.raise_for_status()
        open(p, "wb").write(r.content)
    h = hashlib.sha256(open(p,"rb").read()).hexdigest()
    print(fn, os.path.getsize(p), "bytes sha256", h[:16])  # pin inputs in the report

## 1. Swarm Traces schema (unverified so far: inspect before assuming any field)

In [ ]:
def read_jsonl_gz(p):
    with gzip.open(p, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if line:
                try: yield json.loads(line)
                except json.JSONDecodeError: pass
st = list(read_jsonl_gz("data/st_redacted.jsonl.gz"))
print("rows:", len(st))
keys = collections.Counter(k for r in st for k in r.keys())
print(keys.most_common())
for k in keys:
    vals = [r.get(k) for r in st[:5000]]
    nn = sum(v is not None and v != "" for v in vals)
    sample = next((v for v in vals if v not in (None, "")), None)
    print(f"{k:25s} non-null(first5k)={nn:5d} sample={str(sample)[:120]!r}")

In [ ]:
# Categorical-looking fields: value counts
for k in keys:
    vals = [r.get(k) for r in st if not isinstance(r.get(k), (dict, list))]
    nun = len(set(map(str, vals)))
    if 1 < nun <= 50:
        print(k, collections.Counter(map(str, vals)).most_common(15))

In [ ]:
# Pick the main text field: the string field with the largest median length
str_fields = [k for k in keys if sum(isinstance(r.get(k), str) for r in st[:2000]) > 1000]
med = {k: np.median([len(r.get(k) or "") for r in st[:5000]]) for k in str_fields}
print(sorted(med.items(), key=lambda x: -x[1]))
TEXT = max(med, key=med.get)   # override manually if wrong
print("TEXT field =", TEXT)

## 2. Anchor extraction (2026 window)
Range: 2026-05-01 (1777593600) to 2026-07-31 (1785456000). Note: 1720396800 is July 2024, not 2026.
Anchor types, by reliability:
- `jwt_iat`: issued-at of a token minted at runtime (strong).
- `gzip_mtime`: mtime in a base64 gzip header (H4sI...) (medium: may be 0 or inherited).
- `epoch_s` / `epoch_ms`: bare epoch numbers in text (medium).
- `date_str`: human dates like Jul9, 2026-07-09 (weak: often copied comments).
Inherited anchors: a value appearing in several records may date the source, not this record. Keep those as lower bounds only.

In [ ]:
LO, HI = 1777593600, 1785456000
re_epoch_s  = re.compile(r'(?<!\d)(17[78]\d{7})(?!\d)')
re_epoch_ms = re.compile(r'(?<!\d)(17[78]\d{10})(?!\d)')
re_jwt      = re.compile(r'eyJ[A-Za-z0-9_-]{8,}\.(eyJ[A-Za-z0-9_-]{8,})\.[A-Za-z0-9_-]*')
re_gz       = re.compile(r'H4sI[A-Za-z0-9+/]{8,}')
MON = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
re_iso      = re.compile(r'(?<!\d)2026-(0[5-7])-([0-3]\d)(?:[T ]([0-2]\d):([0-5]\d))?')
re_mon      = re.compile(r'(?i)\b(may|jun|jul)[a-z]*\.?\s?([0-3]?\d)\b(?:\s+([0-2]?\d):([0-5]\d))?')

def b64url_json(s):
    s += "=" * (-len(s) % 4)
    try: return json.loads(base64.urlsafe_b64decode(s))
    except Exception: return None

def gzip_mtime(b64):
    try:
        raw = base64.b64decode(b64[:16] + "=" * (-len(b64[:16]) % 4))
        return struct.unpack("<I", raw[4:8])[0]
    except Exception: return None

def anchors(text):
    out = []
    for m in re_jwt.finditer(text):
        p = b64url_json(m.group(1))
        if isinstance(p, dict) and isinstance(p.get("iat"), (int, float)) and LO <= p["iat"] <= HI:
            out.append(("jwt_iat", int(p["iat"])))
    for m in re_gz.finditer(text):
        t = gzip_mtime(m.group(0))
        if t and LO <= t <= HI: out.append(("gzip_mtime", t))
    for m in re_epoch_ms.finditer(text):
        t = int(m.group(1)) // 1000
        if LO <= t <= HI: out.append(("epoch_ms", t))
    for m in re_epoch_s.finditer(text):
        t = int(m.group(1))
        if LO <= t <= HI: out.append(("epoch_s", t))
    for m in re_iso.finditer(text):
        mo, d = int(m.group(1)), int(m.group(2)); hh = int(m.group(3) or 12); mm = int(m.group(4) or 0)
        try: out.append(("date_str", int(dt.datetime(2026, mo, d, hh, mm, tzinfo=dt.timezone.utc).timestamp())))
        except ValueError: pass
    for m in re_mon.finditer(text):
        mo = MON[m.group(1).lower()[:3]]; d = int(m.group(2))
        if not 1 <= d <= 31: continue
        hh = int(m.group(3) or 12); mm = int(m.group(4) or 0)
        try: out.append(("date_str", int(dt.datetime(2026, mo, d, min(hh,23), mm, tzinfo=dt.timezone.utc).timestamp())))
        except ValueError: pass
    return out

rows = []
for i, r in enumerate(st):
    t = r.get(TEXT) or ""
    for kind, ts in anchors(t):
        rows.append((i, kind, ts))
A = pd.DataFrame(rows, columns=["rec", "kind", "ts"])
print("records with >=1 anchor:", A.rec.nunique(), "of", len(st), f"({A.rec.nunique()/len(st):.1%})")
print(A.kind.value_counts())

In [ ]:
# Inheritance check: the same anchor value in many records likely dates a copied source
vc = A.groupby(["kind","ts"]).rec.nunique().rename("n_recs").reset_index()
A = A.merge(vc, on=["kind","ts"])
A["inherited"] = A.n_recs > 1
print(A.groupby(["kind","inherited"]).rec.nunique().unstack(fill_value=0))
A["day"] = pd.to_datetime(A.ts, unit="s", utc=True).dt.date
print(A[~A.inherited].groupby("day").rec.nunique())

## 3. Quick artifact families (masked MinHash LSH) and the Gate 1 check

In [ ]:
from datasketch import MinHash, MinHashLSH
re_redact = re.compile(r'\[[A-Z][A-Z0-9 _\-]{2,}(?: \d+)?\]')
re_mask   = re.compile(r'[A-Za-z0-9+/=_-]{24,}|\b[0-9a-f]{12,}\b|\d+')
def normalize(t):
    t = re_redact.sub(" <RED> ", t)
    t = re_mask.sub(" <N> ", t)
    return re.sub(r'\s+', ' ', t).strip()
def shingles(t, k=5):
    toks = t.split()
    return {" ".join(toks[j:j+k]) for j in range(max(1, len(toks)-k+1))}

anchored = sorted(A.rec.unique())
NPERM = 128
lsh = MinHashLSH(threshold=0.5, num_perm=NPERM)
mh = {}
for i in anchored:
    s = shingles(normalize(st[i].get(TEXT) or ""))
    m = MinHash(num_perm=NPERM)
    for x in s: m.update(x.encode())
    mh[i] = m; lsh.insert(str(i), m)
parent = {i: i for i in anchored}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for i in anchored:
    for j in lsh.query(mh[i]):
        a, b = find(i), find(int(j))
        if a != b: parent[a] = b
fam = {i: find(i) for i in anchored}
A["family"] = A.rec.map(fam)
indep = A[~A.inherited].drop_duplicates("rec")
per_fam = indep.groupby("family").rec.nunique().sort_values(ascending=False)
print("independent anchored records:", len(indep))
print("families with >=15 independent anchors:", (per_fam >= 15).sum())
print(per_fam.head(20))
GATE1 = len(indep) >= 500 and (per_fam >= 15).sum() >= 5
print("GATE 1 PASSED" if GATE1 else "GATE 1 FAILED -> fall back to the wiki-only measurement study (auditor 2 protocol)")

In [ ]:
# Manual audit: read ~20 anchors in context (as text only) to confirm they are real times
for _, row in indep.sample(min(20, len(indep)), random_state=0).iterrows():
    t = st[row.rec].get(TEXT) or ""
    print(row.kind, dt.datetime.fromtimestamp(row.ts, dt.timezone.utc), "| fam", row.family)
    print("   ", normalize(t)[:300].replace("\n", " "))

## 4. collusion.wiki: check whether relation fields encode copying (possible partial ground truth)

In [ ]:
cw = list(read_jsonl_gz("data/cw_revisions.jsonl.gz"))
cwd = pd.DataFrame([{k: v for k, v in r.items() if k != "body"} for r in cw])
print(len(cwd), "revisions")
for k in ["relation_type", "diff_base_reason", "time_grade", "request_action", "wiki"]:
    if k in cwd: print(k, cwd[k].astype(str).value_counts().head(15).to_dict())
print(cwd["uncertainty_seconds"].describe())
ex = next(r for r in cw if r.get("hunks"))
print(json.dumps(ex["hunks"], ensure_ascii=False)[:800])
print("related_event_id non-null:", cwd["related_event_id"].notna().sum())